# DAVID-Net Training — Kaggle

**Pipeline:**
1. Setup (GPU, repo, token)
2. Discover + extract datasets
3. Build manifests
4. **Precompute SSL features** (VideoMAE + WavLM) — one-time, cached
5. **Stage 0 (QACP):** Pretrain on synthetic quadrants (encoders frozen)
6. **Stage 1:** Finetune with real labeled data (init from QACP)
7. Cross-dataset evaluation

**Architecture compliance:**
- Real encoders: VideoMAE-Base + WavLM-Base+
- Cosine schedule with 2-epoch warmup
- Gradient checkpointing on video backbone
- Generator-balanced sampling
- Augmentation: video (JPEG, blur, color) + audio (noise, codec, mask)
- 3 seeds for statistical rigor

Crash-proof via HuggingFace backup.

In [ ]:
# Cell 1: GPU check + install dependencies
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Enable T4 in Settings.")

!pip install -q transformers accelerate scikit-learn jiwer datasets

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

In [ ]:
# Cell 4: Discover mounted datasets
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")

datasets = {}
KNOWN = {
    "fakeavceleb": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    "lav-df": ["aicontentdetections/lav-df"],
    "dfdc-10": ["pranay22077/dfdc-10"],
    "deepfaketimit": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    "celeb-df-v2": ["reubensuju/celeb-df-v2"],
    "asvpoof-2019": ["anishsarkar22/asvpoof-2019-dataset-la"],
    "in-the-wild": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    "wavefake": ["dinaahmed11/wavefake"],
}

for friendly, paths in KNOWN.items():
    for p in paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                datasets[friendly] = candidate
                print(f"  {friendly} -> {candidate}")
                break
        if friendly in datasets:
            break

print(f"\nFound {len(datasets)} datasets.")

In [ ]:
# Cell 5: Extract datasets (skips if already done)
import zipfile

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def extract_if_needed(name, src, dst):
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already done")
        return
    if any(src.rglob("*.mp4")) or any(src.rglob("*.wav")) or any(src.rglob("*.flac")):
        print(f"  {name}: loose files, skipping")
        marker.touch()
        return
    for arch in src.rglob("*.zip"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
        except Exception as e:
            print(f"FAIL: {e}")
    marker.touch()

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    extract_if_needed(name, path, dst)

print("Extraction done.")

In [ ]:
# Cell 6: Build manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    print(f"FakeAVCeleb root: {fakeav_root}")
    !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42
else:
    print("ERROR: FakeAVCeleb not found!")

CONVERTERS = [
    ("dfdc-10", "dfdc-10"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    if root and root.exists():
        !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}

print("\nManifests:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Precompute SSL features (one-time, cached)
import yaml

FAKEAV_MANIFEST = str(MANIFEST_DIR / "fakeavceleb.jsonl")
FAKEAV_ROOT = str(datasets.get("fakeavceleb", ""))
FEAT_CACHE = WORKING / "feature_cache"
FEAT_CACHE.mkdir(exist_ok=True)

# Check if features already cached
n_cached = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
if n_cached > 100:
    print(f"Features already cached: {n_cached} clips. Skipping.")
else:
    print(f"Precomputing SSL features for FakeAVCeleb ({FAKEAV_ROOT})...")
    !cd {REPO} && python -m src.data.extract_features \
        --config {qacp_path if 'qacp_path' in dir() else '/dev/null'} \
        --manifest {FAKEAV_MANIFEST} \
        --out {FEAT_CACHE} \
        --batch-size 4 \
        --root-dir {FAKEAV_ROOT}
    n_cached = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
    print(f"Cached features: {n_cached} clips")

In [ ]:
# Cell 8: QACP Stage 0 config (uses cached features)
QACP_CONFIG = {
    "run_id": "qacp_stage0",
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 12, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None,
    "feature_cache": str(FEAT_CACHE),
    "train_manifest": FAKEAV_MANIFEST,
    "root_dir": FAKEAV_ROOT,
    "modality_dropout": 0.0, "augment": False,
    "batch_size": 8, "num_workers": 2, "epochs": 20,
    "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
    "warmup_epochs": 2, "gradient_checkpointing": True,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
    "seed": 42,
}

qacp_path = WORKING / "qacp_config.yaml"
with open(qacp_path, "w") as f:
    yaml.dump(QACP_CONFIG, f)
print(f"QACP run: {QACP_CONFIG['run_id']}")
print(f"Feature cache: {FEAT_CACHE} ({n_cached} clips)")
print(f"Encoders: {QACP_CONFIG['video_backbone']} + {QACP_CONFIG['audio_backbone']}")

In [ ]:
# Cell 9: Run QACP Stage 0 pretraining
!cd {REPO} && python -m src.training.pretrain_qacp \
    --config {qacp_path} \
    --run-id {QACP_CONFIG['run_id']}

In [ ]:
# Cell 10: Stage 1 config (init from QACP) — 3 seeds
import glob

qacp_ckpt = None
for p in sorted(glob.glob(str(WORKING / "runs/qacp_epoch*.pt")), reverse=True):
    qacp_ckpt = p
    break
if not qacp_ckpt:
    qacp_ckpt = str(WORKING / "runs/qacp_epoch19.pt")
print(f"QACP checkpoint: {qacp_ckpt}")

SEEDS = [42, 123, 456]
STAGE1_CONFIGS = []

for seed in SEEDS:
    cfg = {
        "run_id": f"stage1_seed{seed}",
        "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
        "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
        "video_backbone": "videomae", "audio_backbone": "wavlm",
        "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
        "freeze_blocks": 6, "freeze_feature_extractor": True,
        "init_from": qacp_ckpt,
        "n_frames": 16, "audio_len": 64000, "shard_root": None,
        "feature_cache": str(FEAT_CACHE),
        "train_manifest": FAKEAV_MANIFEST,
        "val_manifest": str(SPLIT_DIR / "fakeavceleb" / "val.jsonl"),
        "root_dir": FAKEAV_ROOT,
        "modality_dropout": 0.15, "augment": True,
        "batch_size": 4, "num_workers": 2, "epochs": 30,
        "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
        "warmup_epochs": 2, "gradient_checkpointing": True,
        "log_every": 10,
        "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
        "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
        "seed": seed,
    }
    path = WORKING / f"stage1_seed{seed}_config.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    STAGE1_CONFIGS.append((seed, path, cfg))
    print(f"Seed {seed}: {cfg['run_id']}")

In [ ]:
# Cell 11: Run Stage 1 training (3 seeds)
for seed, config_path, cfg in STAGE1_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}...")
    print(f"{'='*60}")
    !cd {REPO} && python -m src.training.train \
        --config {config_path} \
        --run-id {cfg['run_id']}

In [ ]:
# Cell 12: Cross-dataset evaluation (all seeds)
import json

eval_datasets = {
    "dfdc-10": MANIFEST_DIR / "dfdc-10.jsonl",
    "celeb-df-v2": MANIFEST_DIR / "celeb-df-v2.jsonl",
    "in-the-wild": MANIFEST_DIR / "in-the-wild.jsonl",
    "wavefake": MANIFEST_DIR / "wavefake.jsonl",
}

all_results = {}
for seed, config_path, stage1_cfg in STAGE1_CONFIGS:
    run_id = stage1_cfg["run_id"]
    best_ckpt = None
    for p in sorted(glob.glob(str(WORKING / f"runs/{run_id}_epoch*.pt")), reverse=True):
        best_ckpt = p
        break
    if not best_ckpt:
        print(f"Seed {seed}: no checkpoint found, skipping")
        continue

    seed_results = {}
    for ds_name, manifest in eval_datasets.items():
        if not manifest.exists():
            continue
        print(f"\nSeed {seed} on {ds_name}...")
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate \
            --config {config_path} \
            --checkpoint {best_ckpt} \
            --manifest {manifest} \
            --out {report_path}
        if report_path.exists():
            with open(report_path) as f:
                r = json.load(f)
            seed_results[ds_name] = {
                "video_auc": r["video"]["auc"],
                "audio_auc": r["audio"]["auc"],
                "quadrant_acc": r["quadrant"]["acc"],
            }
            print(f"  video_auc={r['video']['auc']:.4f} audio_auc={r['audio']['auc']:.4f}")
    all_results[f"seed{seed}"] = seed_results

# Summary
print("\n" + "="*60)
print("CROSS-DATASET SUMMARY")
print("="*60)
for seed_key, ds_results in all_results.items():
    print(f"\n{seed_key}:")
    for ds, m in ds_results.items():
        print(f"  {ds}: v_auc={m['video_auc']:.4f} a_auc={m['audio_auc']:.4f} quad={m['quadrant_acc']:.4f}")

In [ ]:
# Cell 13: Verify HF backup
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
for seed, _, cfg in STAGE1_CONFIGS:
    run = cfg["run_id"]
    try:
        files = list(api.list_repo_tree("MoshinAli/david-net-av-backup",
                                         path_in_repo=f"runs/{run}",
                                         repo_type="model", recursive=True))
        print(f"\n{run} on HF:")
        for f in files:
            if hasattr(f, 'path'): print(f"  {f.path}")
    except Exception as e:
        print(f"  {run}: {e}")